In [4]:
import numpy as np
import matplotlib.pyplot as plt
import importlib
import process_probe as pp
import warnings
warnings.filterwarnings("ignore")

In [10]:
data_original = np.load(r"F:\temp_compare\data_original.npz", allow_pickle=True)
data_modular = np.load(r"F:\temp_compare\data_modular.npz", allow_pickle=True)

In [11]:
X_train_original = data_original['X_trainT']
X_test_original = data_original['X_testT']
y_train_original = data_original['y_trainT']
y_test_original = data_original['y_testT']
X_train_modular = data_modular['X_train']
X_test_modular = data_modular['X_test']
y_train_modular = data_modular['y_train']
y_test_modular = data_modular['y_test']

In [2]:
import torch #type: ignore
import utils
from pathlib import Path
import preprocess_data as ppd
from process_session import session as ss
import process_probe as pp
import process_attribution as pa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

session_id = 1047969464
session_obj = ss(session_id)

spikes_obj = ppd.pre_process_spikes(session_obj.units, session_obj.spike_times, bin_size=0.004, sigma=3)
spikes_obj.getSpkMat(session_obj.active_times[0],session_obj.active_times[1])
spikes_obj.convolve_with_gaussian()
spikes_obj.zscore()

100%|██████████| 922/922 [01:43<00:00,  8.88it/s]


In [3]:
probe_obj = pp.probe(session_obj, session_obj.active_times[0], session_obj.active_times[1])

In [ ]:
lfp_obj = ppd.pre_process_lfp(probe_obj.lfp, 1250)
lfp_obj.filter_lfp(probe_obj.bands)

input_size = spikes_obj.spkMat.shape[1]
hidden_size = 32
num_layers = 1
seqlength = 750
num_epochs = 100
X_train, y_train, X_test, y_test, X_val, y_val = ppd.generate_training_data(lfp_obj, spikes_obj, seqlength)

np.savez_compressed('F:/temp_compare/data.npz', X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, X_val=X_val, y_val=y_val)

# models, lossesAll = utils.train_models(probe_obj, input_size, hidden_size, num_layers, seqlength, device, num_epochs, X_train, y_train)
# output_dir = Path('E:/vbn_s3_cache')
# utils.save_models(models, output_dir, session_id)

# check if a variable models exists otherwise load it from the saved models
# output_dir = Path(r"Z:\Buzsakilabspace\LabShare\NoamNitzan\Open_Access\Allen_2022")
# args = [input_size, hidden_size, num_layers, seqlength, device]
# models = utils.load_models(output_dir, session_id, probe_obj, args)

# output_dir = Path(r'F:\vbn_s3_cache')
# dur = 720
# bin_size = 0.004
# pa.divide_task_for_attr(models, session_id, output_dir, spikes_obj, bin_size, probe_obj, dur)

In [5]:
spikes_obj = ppd.pre_process_spikes(session_obj.units, session_obj.spike_times, bin_size=0.004, sigma=3)
spikes_obj.getSpkMat(session_obj.spontaneous_times[0], session_obj.spontaneous_times[1])
spikes_obj.convolve_with_gaussian()
spikes_obj.zscore()
probe_obj = pp.probe(session_obj, session_obj.spontaneous_times[0], session_obj.spontaneous_times[1])
lfp_obj = ppd.pre_process_lfp(probe_obj.lfp, 1250)
lfp_obj.filter_lfp(probe_obj.bands)

100%|██████████| 8/8 [00:18<00:00,  2.27s/it]


In [7]:
importlib.reload(ppd)

<module 'preprocess_data' from 'e:\\jerry\\github_repo\\spikes2lfp\\preprocess_data.py'>

In [11]:
input_size = spikes_obj.spkMat.shape[1]
hidden_size = 32
num_layers = 1
seqlength = 750
num_epochs = 100
X_train, y_train, X_test, y_test = ppd.generate_training_data(lfp_obj, spikes_obj, seqlength)
X_train = torch.concat([X_train, X_test])
y_train = torch.concat([y_train, y_test])

In [12]:
np.savez_compressed('F:/temp_compare/data_spont.npz', X_train=X_train, y_train=y_train)

In [2]:
import numpy as np
data_original = np.load(r"F:\temp_compare\data_og.npz", allow_pickle=True)
X_train_original = data_original['X_trainT']
X_test_original = data_original['X_testT']
y_train_original = data_original['y_trainT']
y_test_original = data_original['y_testT']

In [3]:
np.allclose(X_train_original, X_train), np.allclose(y_train_original, y_train), np.allclose(X_test_original, X_test), np.allclose(y_test_original, y_test)

(True, True, True, True)

In [ ]:
import torch
X_train_original = torch.Tensor(data_original['X_trainT']).float()
X_test_original = torch.Tensor(data_original['X_testT']).float()
y_train_original = torch.Tensor(data_original['y_trainT']).float()
y_test_original = torch.Tensor(data_original['y_testT']).float()
(X_train_original == X_train).all(), (X_test_original == X_test).all(), (y_train_original == y_train).all(), (y_test_original == y_test).all()

(tensor(True), tensor(True), tensor(True), tensor(True))

In [4]:
import blosc2
import numpy as np

In [78]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band1_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band1.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.00055240013 0.0005524001
73679.13 73679.125
0.09417887 0.0941789
True


In [79]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band2_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band2.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.000684496 0.000684496
91298.08 91298.08
0.020936504 0.020936504
True


In [80]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band3_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band3.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.002583404 0.002583404
344574.44 344574.44
1.8097694 1.8097694
True


In [81]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band4_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band4.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.0010266793 0.0010266793
136938.48 136938.48
0.18416052 0.18416052
True


In [82]:
a_noam = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band5_noam.npy")
a_mine = np.load(r"F:\vbn_s3_cache\attrs\1044385384\attribution_scores_chan0_band5.npy")
print(np.mean(a_noam), np.mean(a_mine))
print(np.sum(a_noam), np.sum(a_mine))
print(np.std(a_noam), np.std(a_mine))
print(np.allclose(np.mean(a_noam, axis=0), np.mean(a_mine, axis=0)))

0.0006470854 0.0006470854
86308.26 86308.26
0.07330191 0.073301904
True
